# Fase 2 — Ensemble Methods, Optimización y SHAP
**Dataset:** Sample - Superstore | **Target:** `Is_Profitable` (clasificación binaria)

---

## 0. Setup

In [ ]:
!pip install xgboost shap --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print('Todo cargado correctamente.')

## 1. Carga y preparación del dataset

In [ ]:
df = pd.read_csv('Sample_-_Superstore.csv', encoding='latin-1')

df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

df['Ship_Days']     = (df['Ship Date'] - df['Order Date']).dt.days
df['Order_Year']    = df['Order Date'].dt.year
df['Order_Month']   = df['Order Date'].dt.month
df['Profit_Margin'] = (df['Profit'] / df['Sales'].replace(0, np.nan)).fillna(0)
df['Unit_Price']    = df['Sales'] / df['Quantity']
df['Is_Profitable'] = (df['Profit'] > 0).astype(int)

drop_cols = ['Row ID','Order ID','Order Date','Ship Date',
             'Customer ID','Customer Name','Product ID','Product Name',
             'Country','Postal Code','City']
df.drop(columns=drop_cols, inplace=True)
df.drop_duplicates(inplace=True)

label_cols = ['Ship Mode','Segment','Region','Category','Sub-Category']
le = LabelEncoder()
for col in label_cols:
    df[col + '_enc'] = le.fit_transform(df[col])

top_states = df['State'].value_counts().nlargest(20).index
df['State_grp'] = df['State'].where(df['State'].isin(top_states), other='Other')
df = pd.get_dummies(df, columns=['State_grp'], drop_first=True)
df.drop(columns=label_cols + ['State'], inplace=True)

TARGET  = 'Is_Profitable'
EXCLUDE = ['Is_Profitable', 'Profit', 'Profit_Margin']
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in EXCLUDE]

X = df[feature_cols].fillna(0)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Features: {len(feature_cols)}')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Balance -> 0 (no rentable): {(y==0).mean():.1%} | 1 (rentable): {(y==1).mean():.1%}')

---
## 2. Justificacion de la eleccion de modelos

El objetivo es **clasificar pedidos como rentables o no rentables** a partir de variables conocidas en el momento de la venta.

| Algoritmo | Rol | Justificacion |
|-----------|-----|---------------|
| Decision Tree | Baseline interpretable | Reglas simples que los stakeholders pueden seguir. Valida que el modelo aprende patrones coherentes de negocio. |
| Random Forest | Bagging robusto | Promedia 200 arboles independientes, reduciendo varianza. Proporciona Feature Importance estable. |
| XGBoost | Boosting de alto rendimiento | Corrige errores secuencialmente. Estado del arte en datos tabulares estructurados. |

La progresion Tree -> Forest -> XGBoost refleja el aumento de complejidad con el arbol como referencia interpretable.

---
## 3. Modelo 1 — Decision Tree (Baseline)

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

acc_dt = accuracy_score(y_test, y_pred_dt)
f1_dt  = f1_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_prob_dt)

print('=== Decision Tree (max_depth=5) ===')
print(classification_report(y_test, y_pred_dt))
print(f'AUC-ROC: {auc_dt:.4f}')

In [ ]:
# Visualizacion del arbol (primeros 3 niveles)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt, feature_names=feature_cols,
          class_names=['No rentable', 'Rentable'],
          filled=True, rounded=True, fontsize=8, ax=ax, max_depth=3)
plt.title('Decision Tree — primeros 3 niveles', fontsize=13)
plt.tight_layout()
plt.savefig('dt_tree.png', bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de confusion y curva ROC
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Matriz de Confusion — Decision Tree')
axes[0].set_xlabel('Predicho'); axes[0].set_ylabel('Real')

fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
axes[1].plot(fpr_dt, tpr_dt, label=f'DT (AUC={auc_dt:.3f})', color='steelblue', lw=2)
axes[1].plot([0,1],[0,1],'k--', lw=0.8)
axes[1].set_title('Curva ROC — Decision Tree')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend()

plt.tight_layout()
plt.savefig('dt_metricas.png', bbox_inches='tight')
plt.show()

### Conclusion Decision Tree

El arbol actua como baseline interpretable. Con `max_depth=5` evitamos el sobreajuste tipico de los arboles sin podar.

- Los primeros nodos de division suelen ser `Discount` y `Unit_Price`, confirmando que el descuento es el principal predictor de rentabilidad.
- Util para explicar decisiones a stakeholders no tecnicos: se puede seguir el camino de un pedido y entender por que se clasifica como no rentable.
- **Limitacion:** el arbol individual es inestable y pierde poder predictivo frente a los modelos ensemble.

---
## 4. Modelo 2 — Random Forest (Bagging)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=10,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = f1_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print('=== Random Forest (200 arboles) ===')
print(classification_report(y_test, y_pred_rf))
print(f'AUC-ROC: {auc_rf:.4f}')

In [ ]:
# Feature Importance
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top20.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top-20 Feature Importance — Random Forest')
axes[0].set_xlabel('Importancia media (Gini)')

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Matriz de Confusion — Random Forest')
axes[1].set_xlabel('Predicho'); axes[1].set_ylabel('Real')

plt.tight_layout()
plt.savefig('rf_metricas.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Top-10 variables mas importantes para predecir rentabilidad:\n')
for i, (feat, imp) in enumerate(importances.nlargest(10).items(), 1):
    print(f'  {i:2}. {feat:<30} {imp:.4f}')

### Conclusion Random Forest + Feature Importance

Random Forest mejora el AUC respecto al arbol individual mediante **bagging**: promedia las predicciones de 200 arboles entrenados sobre muestras bootstrap, reduciendo la varianza.

- `Discount` lidera la importancia: el descuento excesivo es el principal destructor de margen.
- Variables de categoria (`Category_enc`, `Sub-Category_enc`) aparecen en posiciones medias: la rentabilidad varia por linea de producto.
- `Ship_Days` tiene baja importancia: la velocidad de entrega no afecta significativamente a la rentabilidad.
- **Nota:** Feature Importance por Gini puede sobreestimar variables continuas de alta cardinalidad. SHAP (seccion 8) corregira esto.

---
## 5. Modelo 3 — XGBoost (Boosting)

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb  = f1_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

print('=== XGBoost (300 estimadores) ===')
print(classification_report(y_test, y_pred_xgb))
print(f'AUC-ROC: {auc_xgb:.4f}')

In [ ]:
# Curva de aprendizaje
results = xgb.evals_result()
plt.figure(figsize=(8, 4))
plt.plot(results['validation_0']['logloss'], color='darkorange', lw=1.5)
plt.title('Curva de aprendizaje XGBoost (logloss en test)')
plt.xlabel('Numero de arboles'); plt.ylabel('Log Loss')
plt.tight_layout()
plt.savefig('xgb_learning_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de confusion
fig, ax = plt.subplots(figsize=(5, 4))
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', ax=ax)
ax.set_title('Matriz de Confusion — XGBoost')
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout()
plt.savefig('xgb_cm.png', bbox_inches='tight')
plt.show()

### Conclusion XGBoost

XGBoost aplica **boosting**: cada arbol nuevo corrige los errores del conjunto anterior minimizando el gradiente de la funcion de perdida (logloss).

- La curva de aprendizaje muestra convergencia estable sin sobreajuste, gracias a la regularizacion L1/L2 y al `subsample`.
- Tipicamente obtiene el mayor AUC de los tres modelos en datos tabulares.
- Menos interpretable directamente que el arbol, pero el analisis SHAP compensara esta limitacion.

---
## 6. Comparativa de los 3 modelos

In [ ]:
comparativa = pd.DataFrame({
    'Modelo':   ['Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy': [acc_dt,  acc_rf,  acc_xgb],
    'F1-Score': [f1_dt,   f1_rf,   f1_xgb],
    'AUC-ROC':  [auc_dt,  auc_rf,  auc_xgb],
    'Tipo':     ['Baseline', 'Bagging', 'Boosting']
}).set_index('Modelo').round(4)

print(comparativa)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras de metricas
comparativa[['Accuracy','F1-Score','AUC-ROC']].plot(
    kind='bar', ax=axes[0], colormap='Set2', rot=15)
axes[0].set_title('Comparativa de metricas')
axes[0].set_ylim(0.5, 1.0)
axes[0].legend(loc='lower right')

# Curvas ROC
for (label, prob), color in zip(
    [('Decision Tree', y_prob_dt),
     ('Random Forest', y_prob_rf),
     ('XGBoost',       y_prob_xgb)],
    ['steelblue','seagreen','darkorange']):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    axes[1].plot(fpr, tpr, label=f'{label} ({auc:.3f})', color=color, lw=2)

axes[1].plot([0,1],[0,1],'k--', lw=0.8, label='Random')
axes[1].set_title('Curvas ROC comparativas')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('comparativa_modelos.png', bbox_inches='tight')
plt.show()

---
## 7. GridSearchCV — Optimizacion del mejor modelo (XGBoost)

In [ ]:
# Grid reducido para ejecucion razonable (~5-10 min en Colab)
param_grid = {
    'n_estimators':  [200, 400],
    'max_depth':     [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_base = XGBClassifier(
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'Mejores hiperparametros: {grid_search.best_params_}')
print(f'Mejor AUC-ROC en CV:    {grid_search.best_score_:.4f}')

In [ ]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

acc_best = accuracy_score(y_test, y_pred_best)
f1_best  = f1_score(y_test, y_pred_best)
auc_best = roc_auc_score(y_test, y_prob_best)

print('=== XGBoost Optimizado (GridSearchCV) ===')
print(classification_report(y_test, y_pred_best))
print(f'AUC-ROC: {auc_best:.4f}')
print(f'\nXGBoost base:       AUC={auc_xgb:.4f} | F1={f1_xgb:.4f}')
print(f'XGBoost optimizado: AUC={auc_best:.4f} | F1={f1_best:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cv_results = pd.DataFrame(grid_search.cv_results_)
axes[0].hist(cv_results['mean_test_score'], bins=12, color='steelblue', edgecolor='white')
axes[0].axvline(grid_search.best_score_, color='red', ls='--', lw=2,
                label=f'Mejor: {grid_search.best_score_:.4f}')
axes[0].set_title('Distribucion AUC-ROC en GridSearchCV')
axes[0].set_xlabel('AUC-ROC medio (CV)')
axes[0].legend()

cm_best = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Matriz de Confusion — XGBoost Optimizado')
axes[1].set_xlabel('Predicho'); axes[1].set_ylabel('Real')

plt.tight_layout()
plt.savefig('gridsearch_resultados.png', bbox_inches='tight')
plt.show()

---
## 8. Analisis SHAP — Explicabilidad del modelo

In [ ]:
# Muestra de 500 obs para agilizar el calculo
X_shap = X_test.sample(500, random_state=42)

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_shap)

print(f'Valores SHAP calculados para {len(X_shap)} observaciones.')

### Grafico SHAP 1 — Summary Plot (importancia global)

In [ ]:
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, X_shap, plot_type='bar',
                  max_display=15, show=False)
plt.title('SHAP — Importancia global (mean |SHAP|)', fontsize=12)
plt.tight_layout()
plt.savefig('shap_summary_bar.png', bbox_inches='tight')
plt.show()

**Reflexion SHAP Summary (barras):**

Este grafico muestra la importancia media absoluta de cada variable sobre todas las predicciones. A diferencia del Feature Importance de Random Forest (basado en Gini), SHAP mide la contribucion real de cada variable al cambio en la prediccion, sin sesgo por cardinalidad. Las variables en la cima son las que mas determinan si un pedido sera clasificado como rentable o no.

### Grafico SHAP 2 — Beeswarm Plot (direccion del efecto)

In [ ]:
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, X_shap, max_display=15, show=False)
plt.title('SHAP — Beeswarm: direccion e intensidad del efecto', fontsize=12)
plt.tight_layout()
plt.savefig('shap_beeswarm.png', bbox_inches='tight')
plt.show()

**Reflexion SHAP Beeswarm:**

Cada punto es una observacion. El color indica el valor de la variable (rojo = alto, azul = bajo) y la posicion en X indica si empuja la prediccion hacia rentable (derecha) o no rentable (izquierda).

- Si `Discount` en rojo aparece a la izquierda: descuentos altos reducen la probabilidad de rentabilidad, lo que confirma la hipotesis del EDA.
- Si `Sales` en rojo aparece a la derecha: pedidos de mayor importe tienden a ser mas rentables.
- Variables de categoria con alta dispersion revelan subcategorias con rentabilidad muy dispar, lo que justifica una revision del pricing por linea de producto.

### Grafico SHAP 3 — Waterfall Plot (explicacion individual)

In [ ]:
# Seleccionamos el primer caso no rentable de la muestra SHAP
y_shap_labels = y_test.loc[X_shap.index].reset_index(drop=True)
caso_idx = y_shap_labels[y_shap_labels == 0].index[0]

shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    shap_values[caso_idx],
    feature_names=feature_cols,
    max_display=12,
    show=False
)
plt.title('SHAP Waterfall — Explicacion de un pedido NO rentable', fontsize=11)
plt.tight_layout()
plt.savefig('shap_waterfall.png', bbox_inches='tight')
plt.show()

**Reflexion SHAP Waterfall:**

El waterfall plot explica una prediccion individual mostrando como cada variable empuja la prediccion desde el valor base (probabilidad media del modelo) hasta la prediccion final.

- Barras rojas: variables que aumentan la probabilidad de ser rentable.
- Barras azules: variables que la reducen.

Este es el grafico que mostrariamos a un comercial para explicar por que un pedido concreto ha sido marcado como de riesgo. Por ejemplo: 'este pedido tiene un descuento del 45% y pertenece a la subcategoria Tables (con perdidas historicas), por eso el modelo lo clasifica como no rentable'. Convierte el modelo en una herramienta accionable para el equipo de ventas.

---
## 9. Resumen final y Recomendaciones de negocio

In [ ]:
resumen_final = pd.DataFrame({
    'Modelo':   ['Decision Tree','Random Forest','XGBoost','XGBoost Optimizado'],
    'Accuracy': [acc_dt, acc_rf, acc_xgb, acc_best],
    'F1-Score': [f1_dt,  f1_rf,  f1_xgb,  f1_best],
    'AUC-ROC':  [auc_dt, auc_rf, auc_xgb, auc_best]
}).set_index('Modelo').round(4)

print('=== RESUMEN FINAL DE METRICAS ===')
print(resumen_final)

### Recomendaciones de negocio justificadas con los resultados

---

**1. Implementar el modelo en el flujo de aprobacion de pedidos**

El XGBoost optimizado predice la rentabilidad antes de confirmar el pedido. Integrarlo como alerta en el CRM permitiria al equipo comercial revisar pedidos de alto riesgo antes de enviarlos. Impacto estimado: reduccion del porcentaje de pedidos no rentables procesados.

---

**2. Redisenar la politica de descuentos** *(fuente: SHAP Beeswarm + Feature Importance)*

`Discount` es la variable con mayor impacto negativo en la rentabilidad segun SHAP. Propuesta: establecer un umbral maximo del 20% para pedidos de Furniture y Tables, con aprobacion de manager para superarlo. Esto alinea la politica comercial con la evidencia del modelo.

---

**3. Revisar el pricing de Tables y Bookcases** *(fuente: Feature Importance RF + EDA)*

Estas subcategorias aparecen como predictores negativos recurrentes. El problema puede ser de coste de producto, devoluciones elevadas o descuentos sistematicos. Recomendacion: analisis de margen por referencia especifica dentro de estas categorias.

---

**4. Enfocar la captacion en el segmento Corporate** *(fuente: EDA + SHAP)*

Corporate presenta mejor margen por pedido que Consumer. El modelo confirma que `Segment_enc` tiene impacto positivo para valores corporativos. Recomendacion: redirigir parte del presupuesto de marketing de Consumer a Corporate y Home Office.

---

**5. Monitorizar el modelo en produccion**

Las variables mas importantes son susceptibles a cambios de mercado. Recomendacion: reentrenar trimestralmente y monitorizar el AUC con un dashboard de drift de datos para detectar degradacion del modelo a tiempo.